In [1]:
from transformers import AutoModelForCausalLM
import torch
from huggingface_hub import login

login()

In [2]:
model_id = "google/gemma-3-1b-it"

hf_model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto",
).eval()


In [3]:
print(hf_model)

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((1152,), e

## Try my version

In [4]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

import torch

from src.gemma3_llm import Gemma3ForCausalLM

model_config = {
        'vocab_size': 262144,
        'residual_channel_size': 1152,
        'num_hidden_layers': 26,
        'intermediate_size': 6912,
        'query_dim': 1024,
        'total_kv_dim': 256,
        'head_dimension': 256
    }

my_model = Gemma3ForCausalLM(model_config)
print(my_model)

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(
      (embedding): Embedding(262144, 1152, padding_idx=0)
    )
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (output_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm()
          (k_norm): Gemma3RMSNorm()
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma3RMSN

In [5]:
# Get computational graph of a forward pass
model = my_model
calls = []

def hook(module, inp, out):
    # record module's qualified name if present
    calls.append(module.__class__.__name__ + ":" + (getattr(module, "_get_name", lambda: "")()))

hooks = []
for name, module in model.named_modules():
    h = module.register_forward_hook(lambda m, i, o, n=name: calls.append(n))
    hooks.append(h)

sample = torch.randint(0, 1000, (1, 16)).to(next(model.parameters()).device)  # adjust vocab range & dtype
_ = model(sample)

# cleanup hooks
for h in hooks:
    h.remove()

for i, c in enumerate(calls[:200]):
    print(i, c)

0 model.embed_tokens.embedding
1 model.embed_tokens
2 model.layers.0.input_layernorm
3 model.layers.0.self_attn.q_proj
4 model.layers.0.self_attn.k_proj
5 model.layers.0.self_attn.v_proj
6 model.layers.0.self_attn.q_norm
7 model.layers.0.self_attn.k_norm
8 model.rotary_emb
9 model.rotary_emb
10 model.layers.0.self_attn.output_proj
11 model.layers.0.self_attn
12 model.layers.0.post_attention_layernorm
13 model.layers.0.pre_feedforward_layernorm
14 model.layers.0.mlp.gate_proj
15 model.layers.0.mlp.up_proj
16 model.layers.0.mlp.act_fn
17 model.layers.0.mlp.down_proj
18 model.layers.0.mlp
19 model.layers.0.post_feedforward_layernorm
20 model.layers.0
21 model.layers.1.input_layernorm
22 model.layers.1.self_attn.q_proj
23 model.layers.1.self_attn.k_proj
24 model.layers.1.self_attn.v_proj
25 model.layers.1.self_attn.q_norm
26 model.layers.1.self_attn.k_norm
27 model.rotary_emb
28 model.rotary_emb
29 model.layers.1.self_attn.output_proj
30 model.layers.1.self_attn
31 model.layers.1.post_atte

In [6]:
num_params = sum(p.numel() for p in hf_model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {num_params / 1e9:.3f} Billion")

dummy_input = torch.randint(low=0, high=model_config['vocab_size'], size=(2, 10))
print(f"Running Forward Pass with Input Shape: {dummy_input.shape}")

with torch.no_grad():
    logits = my_model(dummy_input)

print(f"Output Logits Shape: {logits.shape}")
assert logits.shape == (2, 10, model_config['vocab_size'])
print("Model assembled and forward pass successful!")

Total Trainable Parameters: 1.000 Billion
Running Forward Pass with Input Shape: torch.Size([2, 10])
Output Logits Shape: torch.Size([2, 10, 262144])
Model assembled and forward pass successful!


In [7]:
import collections

hf_state_dict = hf_model.state_dict()
my_state_dict = my_model.state_dict()

print(f"\nOfficial model has {len(hf_state_dict)} tensors.")
print(f"Custom model has {len(my_state_dict)} tensors.")

new_state_dict = collections.OrderedDict()
for hf_key, hf_tensor in hf_state_dict.items():
    my_key = hf_key
    
    if my_key in my_state_dict:
        if my_state_dict[my_key].shape == hf_tensor.shape:
            new_state_dict[my_key] = hf_tensor
        else:
            print(f"Shape mismatch for key: {my_key}. HF: {hf_tensor.shape}, My: {my_state_dict[my_key].shape}")
    else:
        # I changed these layers name
        if my_key.endswith("o_proj.weight"):
            new_state_dict['.'.join(my_key.split('.')[:-2])+".output_proj.weight"] = hf_tensor
        elif my_key == "model.embed_tokens.weight":
            new_state_dict['.'.join(my_key.split('.')[:-1])+".embedding.weight"] = hf_tensor   
        else:
            print(f"Key not found in custom model: {my_key}")

print("Loading weights into custom model")
report = my_model.load_state_dict(new_state_dict, strict=False)

if not report.missing_keys and not report.unexpected_keys:
    print("Weights successfully loaded")
else:
    if report.missing_keys:
        print("Missing keys in custom model:", report.missing_keys)
    if report.unexpected_keys:
        print("Unexpected keys from HF model:", report.unexpected_keys)


Official model has 341 tensors.
Custom model has 341 tensors.
Loading weights into custom model
Weights successfully loaded


In [8]:
print("Running Verification")
device = "cuda" if torch.cuda.is_available() else "cpu"
hf_model.to(device)
my_model.to(device)

dummy_input = torch.randint(low=0, high=model_config['vocab_size'], size=(1, 15), device=device)

with torch.no_grad():
    hf_output = hf_model(dummy_input).logits
    my_output = my_model(dummy_input)


are_close = torch.allclose(hf_output, my_output, atol=1e-3)

if are_close:
    print("\nVerification successful! Your model is a correct implementation.")
else:
    print("\nVerification failed. Outputs do not match.")
    
    diff = torch.max(torch.abs(hf_output - my_output))
    print(f"Max difference between outputs: {diff.item()}")

Running Verification

Verification failed. Outputs do not match.
Max difference between outputs: 28.985549926757812


In [37]:
# Inspection configuration
dummy_input = torch.randint(low=0, high=model_config['vocab_size'], size=(1, 15), device=device)
n_layer = 0
my_prefix =  my_model.model.layers[n_layer].input_layernorm
hf_prefix = hf_model.model.layers[n_layer].input_layernorm
hf_layer_name = "input_layernorm"
my_layer_name = "input_layernorm"
device = "cuda" if torch.cuda.is_available() else "cpu"

In [43]:
outputs = {"my_model": {}, "hf_model": {}}

# Hook to store the output at each layer
def get_hook(name, model_type):
    def hook(model, input, output):
        if isinstance(output, tuple):
            outputs[model_type][name] = output[0].detach()
        else:
            outputs[model_type][name] = output.detach()
    return hook

my_norm_hook = my_prefix.register_forward_hook(get_hook(my_layer_name, "my_model"))
hf_norm_hook = hf_prefix.register_forward_hook(get_hook(hf_layer_name, "hf_model"))

with torch.no_grad():
    hf_model(dummy_input)
    my_model(dummy_input)

are_attn_inputs_close = torch.allclose(
    outputs["my_model"][my_layer_name], 
    outputs["hf_model"][hf_layer_name], 
    atol=1e-4
)
print(f"Are the inputs to the attention block (i.e., outputs of input_layernorm) close? -> {are_attn_inputs_close}")

if not are_attn_inputs_close:
    diff = torch.max(torch.abs(outputs["my_model"][my_layer_name] - outputs["hf_model"][hf_layer_name]))
    print(f"  - Max difference at {my_layer_name}: {diff.item()}")

# --- Remove hooks ---
my_norm_hook.remove()
hf_norm_hook.remove()

Are the inputs to the attention block (i.e., outputs of input_layernorm) close? -> False
  - Max difference at input_layernorm: 7.281618118286133
